In [1]:
!node --version
!npm --version

v22.23.4
10.8.2


In [2]:
!npm install -g @sap/cds-dk
!cds --version

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 1 package in 5s
⠴
⠴36 packages are looking for funding
⠴  run `npm fund` for details
⠴
  @sap/cds-dk (global)  10.2.1   /tools/node/lib/node_modules/@sap/cds-dk                       
  cds.home                       /tools/node/lib/node_modules/@sap/cds-dk/node_modules/@sap/cds 
  cds.root                       /content                                                       
  npm root -l                    ./node_modules                                                 
  npm root -g                    /tools/node/lib/node_modules                                   
  Node.js               22.23.4  /tools/node/bin/node                                           
  npm                   10.8.2   /tools/node/bin/npm                                            



In [3]:
!cds init procurement-management
%cd /content/procurement-management
!npm install


Successfully initialized CAP project without a runtime. Choose one with cds add nodejs or cds add java.
Continue with: code procurement-management 

/content/procurement-management
⠙npm error code ENOENT
⠙npm error syscall open
⠙npm error path /content/procurement-management/package.json
⠙npm error errno -2
⠙npm error enoent Could not read package.json: Error: ENOENT: no such file or directory, open '/content/procurement-management/package.json'
⠙npm error enoent This is related to npm not being able to find a file.
npm error enoent
⠙npm error A complete log of this run can be found in: /root/.npm/_logs/2026-11-21T15_09_51_138Z-debug-0.log
⠙

In [4]:
%%writefile db/schema.cds
namespace procurement;

entity Suppliers {
    key ID : UUID;
    name : String(100);
    email : String(100);
}

entity PurchaseOrders {
    key ID : UUID;
    supplier : Association to Suppliers;
    material : String(100);
    quantity : Integer;
    unitPrice : Decimal(10,2);
    status : String(30);
}

entity GoodsReceipts {
    key ID : UUID;
    purchaseOrder : Association to PurchaseOrders;
    receivedQuantity : Integer;
    receiptDate : Date;
}

Writing db/schema.cds


In [5]:
%%writefile srv/procurement-service.cds
using procurement from '../db/schema';

service ProcurementService {
    entity Suppliers as projection on procurement.Suppliers;
    entity PurchaseOrders as projection on procurement.PurchaseOrders;
    entity GoodsReceipts as projection on procurement.GoodsReceipts;
}

Writing srv/procurement-service.cds


In [6]:
!npm install --save-dev @cap-js/sqlite
!npx cds deploy --to sqlite:db.sqlite
!npx cds compile srv/procurement-service.cds --to edmx

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 72 packages in 6s
⠸
⠸33 packages are looking for funding
⠸  run `npm fund` for details
⠸⠙/> successfully deployed to db.sqlite 

(node:1287) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
⠙⠙<?xml version="1.0" encoding="utf-8"?>
<edmx:Edmx Version="4.0" xmlns:edmx="http://docs.oasis-open.org/odata/ns/edmx">
  <edmx:Reference Uri="https://sap.github.io/odata-vocabularies/vocabularies/Common.xml">
    <edmx:Include Alias="Common" Namespace="com.sap.vocabularies.Common.v1"/>
  </edmx:Reference>
  <edmx:Reference Uri="https://oasis-tcs.github.io/odata-vocabularies/vocabularies/Org.OData.Core.V1.xml">
    <edmx:Include Alias="Core" Namespace="Org.OData.Core.V1"/>
  </edmx:Reference>
  <edmx:DataServices>
    <Schema Namespace="ProcurementService" xmlns="http://docs.oasis-open.org/odata/ns/edm">
      <Ann

In [7]:
import sqlite3
import uuid

conn = sqlite3.connect("/content/procurement-management/db.sqlite")
cursor = conn.cursor()

supplier_id = str(uuid.uuid4())
po_id = str(uuid.uuid4())
receipt_id = str(uuid.uuid4())

cursor.execute("""
INSERT INTO procurement_Suppliers (ID, name, email)
VALUES (?, ?, ?)
""", (supplier_id, "Tech Solutions", "sales@example.com"))

cursor.execute("""
INSERT INTO procurement_PurchaseOrders
(ID, supplier_ID, material, quantity, unitPrice, status)
VALUES (?, ?, ?, ?, ?, ?)
""", (po_id, supplier_id, "Dell Laptop", 20, 3500, "Ordered"))

cursor.execute("""
INSERT INTO procurement_GoodsReceipts
(ID, purchaseOrder_ID, receivedQuantity, receiptDate)
VALUES (?, ?, ?, ?)
""", (receipt_id, po_id, 17, "2026-11-21"))

conn.commit()

print("SAP CAP sample data inserted successfully!")

SAP CAP sample data inserted successfully!


In [8]:
import pandas as pd

query = """
SELECT
    po.material,
    po.quantity AS ordered,
    gr.receivedQuantity AS received,
    po.quantity - gr.receivedQuantity AS missing,
    po.unitPrice,
    (po.quantity - gr.receivedQuantity) * po.unitPrice
        AS missing_value
FROM procurement_PurchaseOrders po
JOIN procurement_GoodsReceipts gr
ON po.ID = gr.purchaseOrder_ID
"""

df = pd.read_sql_query(query, conn)

df["status"] = df["missing"].apply(
    lambda x: "Partial Delivery" if x > 0 else "Completed"
)

display(df)

,material,ordered,received,missing,unitPrice,missing_value,status
0,Dell Laptop,20,17,3,3500.0,10500.0,Partial Delivery


In [9]:
%%writefile /content/procurement-management/srv/procurement-service.js
module.exports = (srv) => {

  const { PurchaseOrders, GoodsReceipts } = srv.entities;

  srv.before('CREATE', 'GoodsReceipts', async (req) => {

    const { purchaseOrder_ID, receivedQuantity } = req.data;

    if (!purchaseOrder_ID || !Number.isInteger(receivedQuantity) ||
        receivedQuantity <= 0) {
      return req.reject(400, 'Invalid receipt data');
    }

    const tx = srv.tx(req);

    const order = await tx.run(
      SELECT.one.from(PurchaseOrders)
        .where({ ID: purchaseOrder_ID })
    );

    if (!order) {
      return req.reject(404, 'Purchase order not found');
    }

    const receipts = await tx.run(
      SELECT.from(GoodsReceipts)
        .columns('receivedQuantity')
        .where({ purchaseOrder_ID })
    );

    const totalReceived = receipts.reduce(
      (sum, receipt) => sum + receipt.receivedQuantity, 0
    );

    if (totalReceived + receivedQuantity > order.quantity) {
      return req.reject(
        400,
        'Goods receipt exceeds ordered quantity'
      );
    }

  });

};

Writing /content/procurement-management/srv/procurement-service.js


In [10]:
import os
import subprocess
import time

project = "/content/procurement-management"

server = subprocess.Popen(
    ["npx", "cds", "serve", "--from", "srv", "--port", "4004"],
    cwd=project,
    stdout=open("/content/cap-server.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Server process:", server.poll())
print(open("/content/cap-server.log").read()[-2000:])

Server process: 1
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> deployment to in-memory database. failed

[ERROR] Expected option ‘service’ to have ‘type string’. Found: ‘type undefined’



In [11]:
import requests

url = "http://localhost:4004/odata/v4/procurement/PurchaseOrders"

response = requests.get(url)

print("Status:", response.status_code)
print(response.text[:1500])

ConnectionError: HTTPConnectionPool(host='localhost', port=4004): Max retries exceeded with url: /odata/v4/procurement/PurchaseOrders (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7db5db4b46e0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [12]:
import os

project = "/content/procurement-management"

print("Project exists:", os.path.isdir(project))
print("Database exists:", os.path.isfile(project + "/db.sqlite"))
print("Service exists:", os.path.isfile(
    project + "/srv/procurement-service.cds"
))

Project exists: True
Database exists: True
Service exists: True


In [13]:
import subprocess
import time
import requests

project = "/content/procurement-management"

# Start SAP CAP server
log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

# Wait for server startup
time.sleep(8)

print("Server running:", server.poll() is None)

log.flush()
print(open("/content/cap-server.log").read()[-3000:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 693 ms
(node:2282) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [14]:
import requests

url = "http://127.0.0.1:4004"

try:
    response = requests.get(url, timeout=10)
    print("Status:", response.status_code)
    print(response.text[:1000])

except requests.exceptions.RequestException as e:
    print("Connection failed:", e)
    print("\nServer log:")
    print(open("/content/cap-server.log").read()[-3000:])

Status: 200
<!DOCTYPE html>
<html lang="en">
<head>
  <title>undefined</title>
  <meta name="color-scheme" content="dark light">
  <link rel="stylesheet" href="/.cap/index.css">
  <script src="/.cap/index-client.js" defer></script>
  <script> document.app_info = {"webapps":[],"agents":[]} </script>
</head>
<body>

  <label class="theme-switch" title="Toggle dark mode">
    <input id="theme-switch" type="checkbox" aria-label="Toggle dark mode">
    <span class="theme-switch-track" aria-hidden="true"><span class="theme-switch-sun">☀</span><span class="theme-switch-moon">☾</span></span>
  </label>

  <h1> Welcome to <i>@sap/cds</i> Server </h1>
  <p id="subtitle"> Serving <i>undefined</i> </p>


  <section id="webapps">
    <h2> Web Apps: </h2>
    <ul>
      <li><a href="${href}" target="_blank">${label}</a></li>
    </ul>
  </section>


  <section id="agents">
    <h2> Agents: </h2>
    <ul>
      <li>
        <a href="${href}" target="_blank" title="Chat Preview"> ${label} </a>
       

In [15]:
import requests

base_url = "http://127.0.0.1:4004"

url = base_url + "/odata/v4/procurement/PurchaseOrders"

response = requests.get(url, timeout=10)

print("Status:", response.status_code)
print(response.text[:2000])

Status: 200
{"@odata.context":"$metadata#PurchaseOrders","value":[]}


In [16]:
import requests

base_url = "http://127.0.0.1:4004/odata/v4/procurement"

supplier = {
    "name": "Tech Solutions",
    "email": "sales@example.com"
}

response = requests.post(
    base_url + "/Suppliers",
    json=supplier,
    timeout=10
)

print("Status:", response.status_code)
print(response.text[:1500])

Status: 201
{"@odata.context":"$metadata#Suppliers/$entity","ID":"bfcbfedc-a9d2-4d82-8961-4e46c154df8c","name":"Tech Solutions","email":"sales@example.com"}


In [17]:
supplier_data = response.json()

purchase_order = {
    "supplier_ID": supplier_data["ID"],
    "material": "Dell Laptop",
    "quantity": 20,
    "unitPrice": 3500,
    "status": "Ordered"
}

po_response = requests.post(
    base_url + "/PurchaseOrders",
    json=purchase_order,
    timeout=10
)

print("Status:", po_response.status_code)
print(po_response.text[:1500])

Status: 201
{"@odata.context":"$metadata#PurchaseOrders/$entity","ID":"275e0379-6160-45bf-8e58-68336c0c4db4","supplier_ID":"bfcbfedc-a9d2-4d82-8961-4e46c154df8c","material":"Dell Laptop","quantity":20,"unitPrice":"3500.00","status":"Ordered"}


In [18]:
check = requests.get(
    base_url + "/PurchaseOrders",
    timeout=10
)

print("Status:", check.status_code)
print(check.json())

Status: 200
{'@odata.context': '$metadata#PurchaseOrders', 'value': [{'ID': '275e0379-6160-45bf-8e58-68336c0c4db4', 'supplier_ID': 'bfcbfedc-a9d2-4d82-8961-4e46c154df8c', 'material': 'Dell Laptop', 'quantity': 20, 'unitPrice': '3500.00', 'status': 'Ordered'}]}


In [19]:
import requests

base_url = "http://127.0.0.1:4004/odata/v4/procurement"

po_id = "275e0379-6160-45bf-8e58-68336c0c4db4"

goods_receipt = {
    "purchaseOrder_ID": po_id,
    "receivedQuantity": 17,
    "receiptDate": "2026-11-21"
}

response = requests.post(
    base_url + "/GoodsReceipts",
    json=goods_receipt,
    timeout=10
)

print("Status:", response.status_code)
print(response.text[:1500])

Status: 201
{"@odata.context":"$metadata#GoodsReceipts/$entity","ID":"5d8a7da1-c7c1-4395-a1f3-0d2fa252a1e0","purchaseOrder_ID":"275e0379-6160-45bf-8e58-68336c0c4db4","receivedQuantity":17,"receiptDate":"2026-11-21"}


In [20]:
orders = requests.get(
    base_url + "/PurchaseOrders",
    timeout=10
).json()["value"]

receipts = requests.get(
    base_url + "/GoodsReceipts",
    timeout=10
).json()["value"]

for order in orders:
    received = sum(
        r["receivedQuantity"]
        for r in receipts
        if r["purchaseOrder_ID"] == order["ID"]
    )

    missing = order["quantity"] - received

    print("Material:", order["material"])
    print("Ordered:", order["quantity"])
    print("Received:", received)
    print("Missing:", missing)

    if missing > 0:
        print("Status: PARTIAL DELIVERY")
        print(
            "Missing Value:",
            missing * float(order["unitPrice"]),
            "SAR"
        )
    elif missing == 0:
        print("Status: COMPLETED")
    else:
        print("Status: OVER-RECEIPT")

Material: Dell Laptop
Ordered: 20
Received: 17
Missing: 3
Status: PARTIAL DELIVERY
Missing Value: 10500.0 SAR


In [21]:
invalid_receipt = {
    "purchaseOrder_ID": po_id,
    "receivedQuantity": 5,
    "receiptDate": "2026-11-21"
}

test = requests.post(
    base_url + "/GoodsReceipts",
    json=invalid_receipt,
    timeout=10
)

print("Status:", test.status_code)
print(test.text[:1500])

Status: 400
{"error":{"message":"Goods receipt exceeds ordered quantity","code":"400","@Common.numericSeverity":4}}


In [22]:
print("Status:", test.status_code)
print("Error Details:", test.text)

Status: 400
Error Details: {"error":{"message":"Goods receipt exceeds ordered quantity","code":"400","@Common.numericSeverity":4}}


In [23]:
from pathlib import Path

service = Path(
    "/content/procurement-management/srv/procurement-service.js"
)

code = r"""
const cds = require('@sap/cds');

module.exports = (srv) => {

  const { PurchaseOrders, GoodsReceipts, Inventory } = srv.entities;
  const { SELECT, UPDATE, INSERT } = cds.ql;

  srv.before('CREATE', 'GoodsReceipts', async (req) => {

    const { purchaseOrder_ID, receivedQuantity } = req.data;

    if (!purchaseOrder_ID ||
        !Number.isInteger(receivedQuantity) ||
        receivedQuantity <= 0) {
      return req.reject(400, 'Invalid receipt data');
    }

    const tx = cds.tx(req);

    const order = await tx.run(
      SELECT.one.from(PurchaseOrders)
        .where({ ID: purchaseOrder_ID })
    );

    if (!order) {
      return req.reject(404, 'Purchase order not found');
    }

    const receipts = await tx.run(
      SELECT.from(GoodsReceipts)
        .where({ purchaseOrder_ID })
    );

    const totalReceived = receipts.reduce(
      (sum, r) => sum + r.receivedQuantity, 0
    );

    if (totalReceived + receivedQuantity > order.quantity) {
      return req.reject(
        400,
        'Goods receipt exceeds ordered quantity'
      );
    }
  });

  srv.after('CREATE', 'GoodsReceipts', async (data, req) => {

    const tx = cds.tx(req);

    const order = await tx.run(
      SELECT.one.from(PurchaseOrders)
        .where({ ID: data.purchaseOrder_ID })
    );

    const existing = await tx.run(
      SELECT.one.from(Inventory)
        .where({ material: order.material })
    );

    if (existing) {
      await tx.run(
        UPDATE(Inventory)
          .set({
            quantity: existing.quantity + data.receivedQuantity,
            lastUpdated: new Date().toISOString().slice(0, 10)
          })
          .where({ ID: existing.ID })
      );
    } else {
      await tx.run(
        INSERT.into(Inventory).entries({
          ID: cds.utils.uuid(),
          material: order.material,
          quantity: data.receivedQuantity,
          lastUpdated: new Date().toISOString().slice(0, 10)
        })
      );
    }
  });

};
"""

service.write_text(code)

print("SAP Inventory Automation Added!")

SAP Inventory Automation Added!


In [24]:
import subprocess
import time
import requests

project = "/content/procurement-management"

if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=10)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-1500:])

Server running: False
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
(node:3610) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
Error: listen EADDRINUSE: address already in use :::4004
    at Server.setupListenHandle [as _listen2] (node:net:1941:16)
    at listenInCluster (node:net:1998:12)
    at Server.listen (node:net:2103:7)
    at Function.listen (/content/procurement-management/node_modules/express/lib/application.js:605:24)
    at cds_server (/content/procurement-management/node_modules/@sap/cds/server.js:67:27)
    at async Object.serve (

In [25]:
import requests

base_url = "http://127.0.0.1:4004/odata/v4/procurement"

response = requests.get(
    base_url + "/Inventory",
    timeout=10
)

print("Status:", response.status_code)
print(response.text[:1500])

Status: 404
{"error":{"message":"Invalid resource path \"ProcurementService.Inventory\"","code":"404","@Common.numericSeverity":4}}


In [26]:
from pathlib import Path

project = Path("/content/procurement-management")

schema = project / "db/schema.cds"
service = project / "srv/procurement-service.cds"

print("=== DATABASE SCHEMA ===")
print(schema.read_text())

print("\n=== SERVICE DEFINITION ===")
print(service.read_text())

=== DATABASE SCHEMA ===
namespace procurement;

entity Suppliers {
    key ID : UUID;
    name : String(100);
    email : String(100);
}

entity PurchaseOrders {
    key ID : UUID;
    supplier : Association to Suppliers;
    material : String(100);
    quantity : Integer;
    unitPrice : Decimal(10,2);
    status : String(30);
}

entity GoodsReceipts {
    key ID : UUID;
    purchaseOrder : Association to PurchaseOrders;
    receivedQuantity : Integer;
    receiptDate : Date;
}


=== SERVICE DEFINITION ===
using procurement from '../db/schema';

service ProcurementService {
    entity Suppliers as projection on procurement.Suppliers;
    entity PurchaseOrders as projection on procurement.PurchaseOrders;
    entity GoodsReceipts as projection on procurement.GoodsReceipts;
}



In [27]:
from pathlib import Path

project = Path("/content/procurement-management")

schema = project / "db/schema.cds"
service = project / "srv/procurement-service.cds"

# Add Inventory entity
schema_text = schema.read_text()

if "entity Inventory {" not in schema_text:
    schema_text += """

entity Inventory {
    key ID : UUID;
    material : String(100);
    quantity : Integer;
    lastUpdated : Date;
}
"""
    schema.write_text(schema_text)

# Add Inventory OData service
service_text = service.read_text()

if "entity Inventory as projection" not in service_text:
    service_text = service_text.replace(
        "service ProcurementService {",
        """service ProcurementService {
    entity Inventory as projection on procurement.Inventory;"""
    )
    service.write_text(service_text)

print("Inventory entity added successfully!")
print("Inventory OData service added successfully!")

Inventory entity added successfully!
Inventory OData service added successfully!


In [28]:
import shutil
import subprocess

project = "/content/procurement-management"

shutil.copy2(
    project + "/db.sqlite",
    project + "/db-backup.sqlite"
)

result = subprocess.run(
    ["npx", "cds", "deploy", "--to", "sqlite:db.sqlite"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

/> successfully deployed to db.sqlite 


(node:9811) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 0


In [29]:
import subprocess
import time

project = "/content/procurement-management"

# Stop old server
subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

# Start updated server
log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-2000:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 718 ms
(node:9920) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [30]:
import requests

base_url = "http://127.0.0.1:4004/odata/v4/procurement"

response = requests.get(
    base_url + "/Inventory",
    timeout=10
)

print("Status:", response.status_code)
print(response.text[:1500])

Status: 200
{"@odata.context":"$metadata#Inventory","value":[]}


In [31]:
from pathlib import Path
import shutil

project = Path("/content/procurement-management")
js_file = project / "srv/procurement-service.js"

if js_file.exists():
    shutil.copy2(js_file, project / "srv/procurement-service.backup.txt")

print("Backup completed")

Backup completed


In [32]:
from pathlib import Path

js_file = Path(
    "/content/procurement-management/srv/procurement-service.js"
)

code = r"""
const cds = require('@sap/cds');
const { SELECT, INSERT, UPDATE } = cds.ql;

module.exports = (srv) => {

  const { PurchaseOrders, GoodsReceipts, Inventory } = srv.entities;

  srv.on('CREATE', 'GoodsReceipts', async (req, next) => {

    const { purchaseOrder_ID, receivedQuantity } = req.data;

    if (!purchaseOrder_ID ||
        !Number.isInteger(receivedQuantity) ||
        receivedQuantity <= 0) {
      return req.reject(400, 'Invalid receipt data');
    }

    const tx = cds.tx(req);

    const order = await tx.run(
      SELECT.one.from(PurchaseOrders)
        .where({ ID: purchaseOrder_ID })
    );

    if (!order) {
      return req.reject(404, 'Purchase order not found');
    }

    const receipts = await tx.run(
      SELECT.from(GoodsReceipts)
        .where({ purchaseOrder_ID })
    );

    const totalReceived = receipts.reduce(
      (sum, r) => sum + r.receivedQuantity, 0
    );

    if (totalReceived + receivedQuantity > order.quantity) {
      return req.reject(
        400,
        'Goods receipt exceeds ordered quantity'
      );
    }

    const result = await next();

    const existing = await tx.run(
      SELECT.one.from(Inventory)
        .where({ material: order.material })
    );

    const today = new Date().toISOString().slice(0, 10);

    if (existing) {
      await tx.run(
        UPDATE(Inventory)
          .set({
            quantity: existing.quantity + receivedQuantity,
            lastUpdated: today
          })
          .where({ ID: existing.ID })
      );
    } else {
      await tx.run(
        INSERT.into(Inventory).entries({
          ID: cds.utils.uuid(),
          material: order.material,
          quantity: receivedQuantity,
          lastUpdated: today
        })
      );
    }

    return result;
  });

};
"""

js_file.write_text(code)
print("Goods Receipt & Inventory automation configured")

Goods Receipt & Inventory automation configured


In [33]:
import subprocess
import time

project = "/content/procurement-management"

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-2000:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 676 ms
(node:10285) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [34]:
import requests
from datetime import date

base = "http://127.0.0.1:4004/odata/v4/procurement"

# Create a new test purchase order
supplier = requests.get(
    base + "/Suppliers", timeout=10
).json()["value"][0]

po = requests.post(
    base + "/PurchaseOrders",
    json={
        "supplier_ID": supplier["ID"],
        "material": "Test Monitor",
        "quantity": 10,
        "unitPrice": 800,
        "status": "Ordered"
    },
    timeout=10
)

print("Purchase Order:", po.status_code)
print(po.text[:500])

if po.status_code == 201:
    po_id = po.json()["ID"]

    receipt = requests.post(
        base + "/GoodsReceipts",
        json={
            "purchaseOrder_ID": po_id,
            "receivedQuantity": 5,
            "receiptDate": date.today().isoformat()
        },
        timeout=10
    )

    print("Goods Receipt:", receipt.status_code)
    print(receipt.text[:500])

    inventory = requests.get(
        base + "/Inventory", timeout=10
    )

    print("Inventory:", inventory.status_code)
    print(inventory.text[:1000])

IndexError: list index out of range

In [35]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

# Get existing suppliers
response = requests.get(base + "/Suppliers", timeout=10)
response.raise_for_status()

suppliers = response.json()["value"]

if len(suppliers) == 0:

    new_supplier = {
        "name": "DXC Demo Technology Supplier",
        "email": "demo@example.com"
    }

    response = requests.post(
        base + "/Suppliers",
        json=new_supplier,
        timeout=10
    )

    print("Create Supplier Status:", response.status_code)
    print(response.text[:1000])
    response.raise_for_status()

    supplier = response.json()

else:
    supplier = suppliers[0]
    print("Existing supplier selected")

print("Supplier ID:", supplier["ID"])
print("Supplier Name:", supplier["name"])

Create Supplier Status: 201
{"@odata.context":"$metadata#Suppliers/$entity","ID":"3a5d5e83-e1ac-4fd8-bb4a-63f750e91edc","name":"DXC Demo Technology Supplier","email":"demo@example.com"}
Supplier ID: 3a5d5e83-e1ac-4fd8-bb4a-63f750e91edc
Supplier Name: DXC Demo Technology Supplier


In [36]:
purchase_order = {
    "supplier_ID": supplier["ID"],
    "material": "Test Monitor",
    "quantity": 10,
    "unitPrice": 800,
    "status": "Ordered"
}

po_response = requests.post(
    base + "/PurchaseOrders",
    json=purchase_order,
    timeout=10
)

print("Purchase Order Status:", po_response.status_code)
print(po_response.text[:1000])

po_response.raise_for_status()
po_id = po_response.json()["ID"]

Purchase Order Status: 201
{"@odata.context":"$metadata#PurchaseOrders/$entity","ID":"4a504e67-5e9e-4ebf-8804-f18a40438631","supplier_ID":"3a5d5e83-e1ac-4fd8-bb4a-63f750e91edc","material":"Test Monitor","quantity":10,"unitPrice":"800.00","status":"Ordered"}


In [37]:
from datetime import date

goods_receipt = {
    "purchaseOrder_ID": po_id,
    "receivedQuantity": 5,
    "receiptDate": date.today().isoformat()
}

gr_response = requests.post(
    base + "/GoodsReceipts",
    json=goods_receipt,
    timeout=10
)

print("Goods Receipt Status:", gr_response.status_code)
print(gr_response.text[:1000])

Goods Receipt Status: 201
{"@odata.context":"$metadata#GoodsReceipts/$entity","ID":"1cdbe3cb-fbc6-4eef-b4dc-c1c57683c511","purchaseOrder_ID":"4a504e67-5e9e-4ebf-8804-f18a40438631","receivedQuantity":5,"receiptDate":"2026-11-21"}


In [38]:
inventory_response = requests.get(
    base + "/Inventory",
    timeout=10
)

print("Inventory Status:", inventory_response.status_code)
print(inventory_response.text[:1500])

Inventory Status: 200
{"@odata.context":"$metadata#Inventory","value":[{"ID":"ba77db03-df8b-4e34-9c35-3d561065bad6","material":"Test Monitor","quantity":5,"lastUpdated":"2026-11-21"}]}


In [39]:
from pathlib import Path

project = Path("/content/procurement-management")
schema = project / "db/schema.cds"
service = project / "srv/procurement-service.cds"

schema_text = schema.read_text()

if "entity SupplierInvoices {" not in schema_text:
    schema_text += """

entity SupplierInvoices {
    key ID : UUID;
    purchaseOrder : Association to PurchaseOrders;
    invoiceNumber : String(50);
    invoiceQuantity : Integer;
    invoiceAmount : Decimal(12,2);
    status : String(30);
}
"""
    schema.write_text(schema_text)

service_text = service.read_text()

if "entity SupplierInvoices as projection" not in service_text:
    service_text = service_text.replace(
        "service ProcurementService {",
        """service ProcurementService {
    entity SupplierInvoices as projection on procurement.SupplierInvoices;"""
    )
    service.write_text(service_text)

print("SupplierInvoices model added successfully")

SupplierInvoices model added successfully


In [40]:
import shutil
import subprocess
from pathlib import Path

project = Path("/content/procurement-management")

shutil.copy2(
    project / "db.sqlite",
    project / "db-before-invoices.sqlite"
)

result = subprocess.run(
    ["npx", "cds", "deploy", "--to", "sqlite:db.sqlite"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

/> successfully deployed to db.sqlite 


(node:14510) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 0


In [41]:
import subprocess
import time

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd="/content/procurement-management",
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-1500:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 737 ms
(node:14623) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [42]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

response = requests.get(
    base + "/SupplierInvoices",
    timeout=10
)

print("Invoice API Status:", response.status_code)
print(response.text[:1000])

Invoice API Status: 200
{"@odata.context":"$metadata#SupplierInvoices","value":[]}


In [43]:
from pathlib import Path
import shutil

file = Path(
    "/content/procurement-management/srv/procurement-service.js"
)

shutil.copy2(file, file.with_suffix(".js.backup"))

code = file.read_text()

invoice_logic = r"""
  srv.before('CREATE', 'SupplierInvoices', async (req) => {
    const { purchaseOrder_ID, invoiceQuantity,
            invoiceAmount } = req.data;

    if (!purchaseOrder_ID ||
        !Number.isInteger(invoiceQuantity) ||
        invoiceQuantity <= 0 ||
        !Number.isFinite(Number(invoiceAmount)) ||
        Number(invoiceAmount) <= 0) {
      return req.reject(400, 'Invalid invoice data');
    }

    const tx = cds.tx(req);

    const order = await tx.run(
      SELECT.one.from(PurchaseOrders)
        .where({ ID: purchaseOrder_ID })
    );

    if (!order) {
      return req.reject(404, 'Purchase order not found');
    }

    const receipts = await tx.run(
      SELECT.from(GoodsReceipts)
        .where({ purchaseOrder_ID })
    );

    const received = receipts.reduce(
      (sum, r) => sum + r.receivedQuantity, 0
    );

    const invoices = await tx.run(
      SELECT.from(SupplierInvoices)
        .where({ purchaseOrder_ID, status: 'Matched' })
    );

    const alreadyInvoiced = invoices.reduce(
      (sum, i) => sum + i.invoiceQuantity, 0
    );

    if (invoiceQuantity > received - alreadyInvoiced) {
      return req.reject(
        400,
        'Invoice quantity exceeds uninvoiced received quantity'
      );
    }

    const expectedAmount =
      Math.round(invoiceQuantity * Number(order.unitPrice) * 100);

    const actualAmount =
      Math.round(Number(invoiceAmount) * 100);

    if (expectedAmount !== actualAmount) {
      return req.reject(
        400,
        'Invoice amount does not match purchase order'
      );
    }

    req.data.status = 'Matched';
  });
"""

if "Invoice quantity exceeds uninvoiced" not in code:
    code = code.replace(
        "const { PurchaseOrders, GoodsReceipts, Inventory } = srv.entities;",
        "const { PurchaseOrders, GoodsReceipts, Inventory, SupplierInvoices } = srv.entities;"
    )

    position = code.rfind("};")
    code = code[:position] + invoice_logic + "\n" + code[position:]
    file.write_text(code)

print("Invoice Matching logic added!")

Invoice Matching logic added!


In [44]:
import subprocess
import time

project = "/content/procurement-management"

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-1500:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 727 ms
(node:14984) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [45]:
import requests
import uuid

base = "http://127.0.0.1:4004/odata/v4/procurement"

orders_response = requests.get(
    base + "/PurchaseOrders", timeout=10
)
orders_response.raise_for_status()

orders = orders_response.json()["value"]

monitors = [
    o for o in orders
    if o["material"] == "Test Monitor"
]

if not monitors:
    raise ValueError("Test Monitor purchase order not found")

po = monitors[-1]

receipts_response = requests.get(
    base + "/GoodsReceipts", timeout=10
)
receipts_response.raise_for_status()

received = sum(
    int(r["receivedQuantity"])
    for r in receipts_response.json()["value"]
    if r["purchaseOrder_ID"] == po["ID"]
)

invoices_response = requests.get(
    base + "/SupplierInvoices", timeout=10
)
invoices_response.raise_for_status()

invoiced = sum(
    int(i["invoiceQuantity"])
    for i in invoices_response.json()["value"]
    if i["purchaseOrder_ID"] == po["ID"]
    and i["status"] == "Matched"
)

available = received - invoiced

print("Received:", received)
print("Already invoiced:", invoiced)
print("Available to invoice:", available)

if available <= 0:
    print("No remaining quantity to invoice. No invoice created.")
else:
    quantity = min(5, available)

    invoice = {
        "purchaseOrder_ID": po["ID"],
        "invoiceNumber": "INV-" + uuid.uuid4().hex[:8],
        "invoiceQuantity": quantity,
        "invoiceAmount": round(
            quantity * float(po["unitPrice"]), 2
        )
    }

    result = requests.post(
        base + "/SupplierInvoices",
        json=invoice,
        timeout=10
    )

    print("Status:", result.status_code)
    print(result.text[:1500])

ValueError: Test Monitor purchase order not found

In [46]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

response = requests.get(
    base + "/PurchaseOrders",
    timeout=10
)

print("Status:", response.status_code)
print("Purchase Orders:", response.text[:3000])

Status: 200
Purchase Orders: {"@odata.context":"$metadata#PurchaseOrders","value":[]}


In [47]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

# 1. Check suppliers
r = requests.get(base + "/Suppliers", timeout=10)
r.raise_for_status()

suppliers = r.json()["value"]

if suppliers:
    supplier = suppliers[0]
else:
    r = requests.post(
        base + "/Suppliers",
        json={
            "name": "DXC Demo Supplier",
            "email": "demo@example.com"
        },
        timeout=10
    )
    r.raise_for_status()
    supplier = r.json()

# 2. Create purchase order
r = requests.post(
    base + "/PurchaseOrders",
    json={
        "supplier_ID": supplier["ID"],
        "material": "Test Monitor",
        "quantity": 10,
        "unitPrice": 800,
        "status": "Ordered"
    },
    timeout=10
)

print("Purchase Order Status:", r.status_code)
print(r.text[:1000])

r.raise_for_status()

po_id = r.json()["ID"]

print("Purchase Order ID:", po_id)

Purchase Order Status: 201
{"@odata.context":"$metadata#PurchaseOrders/$entity","ID":"cff20188-8c7e-402a-8292-fe658c36d674","supplier_ID":"4ec22eec-d8eb-499a-b23d-96a1f539caee","material":"Test Monitor","quantity":10,"unitPrice":"800.00","status":"Ordered"}
Purchase Order ID: cff20188-8c7e-402a-8292-fe658c36d674


In [48]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

# 1. Check suppliers
r = requests.get(base + "/Suppliers", timeout=10)
r.raise_for_status()

suppliers = r.json()["value"]

if suppliers:
    supplier = suppliers[0]
else:
    r = requests.post(
        base + "/Suppliers",
        json={
            "name": "DXC Demo Supplier",
            "email": "demo@example.com"
        },
        timeout=10
    )
    r.raise_for_status()
    supplier = r.json()

# 2. Create purchase order
r = requests.post(
    base + "/PurchaseOrders",
    json={
        "supplier_ID": supplier["ID"],
        "material": "Test Monitor",
        "quantity": 10,
        "unitPrice": 800,
        "status": "Ordered"
    },
    timeout=10
)

print("Purchase Order Status:", r.status_code)
print(r.text[:1000])

r.raise_for_status()

po_id = r.json()["ID"]

print("Purchase Order ID:", po_id)

Purchase Order Status: 201
{"@odata.context":"$metadata#PurchaseOrders/$entity","ID":"4219b0b7-a668-4a8f-8845-689fec32a83d","supplier_ID":"4ec22eec-d8eb-499a-b23d-96a1f539caee","material":"Test Monitor","quantity":10,"unitPrice":"800.00","status":"Ordered"}
Purchase Order ID: 4219b0b7-a668-4a8f-8845-689fec32a83d


In [49]:
import uuid

invoice = requests.post(
    base + "/SupplierInvoices",
    json={
        "purchaseOrder_ID": po_id,
        "invoiceNumber": "INV-" + uuid.uuid4().hex[:8],
        "invoiceQuantity": 5,
        "invoiceAmount": 4000
    },
    timeout=10
)

print("Invoice Status:", invoice.status_code)
print(invoice.text[:1500])

Invoice Status: 400
{"error":{"message":"Invoice quantity exceeds uninvoiced received quantity","code":"400","@Common.numericSeverity":4}}


In [50]:
invalid_invoice = requests.post(
    base + "/SupplierInvoices",
    json={
        "purchaseOrder_ID": po_id,
        "invoiceNumber": "INV-INVALID-" + uuid.uuid4().hex[:6],
        "invoiceQuantity": 10,
        "invoiceAmount": 8000
    },
    timeout=10
)

print("Invalid Invoice Status:", invalid_invoice.status_code)
print(invalid_invoice.text[:1500])

Invalid Invoice Status: 400
{"error":{"message":"Invoice quantity exceeds uninvoiced received quantity","code":"400","@Common.numericSeverity":4}}


In [51]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

entities = [
    "PurchaseOrders",
    "GoodsReceipts",
    "Inventory",
    "SupplierInvoices"
]

for entity in entities:
    response = requests.get(
        f"{base}/{entity}",
        timeout=10
    )

    print(f"\n=== {entity} ===")
    print("Status:", response.status_code)

    if response.ok:
        records = response.json().get("value", [])
        print("Total Records:", len(records))
        print("Sample:", records[:1])
    else:
        print(response.text[:500])


=== PurchaseOrders ===
Status: 200
Total Records: 2
Sample: [{'ID': '4219b0b7-a668-4a8f-8845-689fec32a83d', 'supplier_ID': '4ec22eec-d8eb-499a-b23d-96a1f539caee', 'material': 'Test Monitor', 'quantity': 10, 'unitPrice': '800.00', 'status': 'Ordered'}]

=== GoodsReceipts ===
Status: 200
Total Records: 0
Sample: []

=== Inventory ===
Status: 200
Total Records: 0
Sample: []

=== SupplierInvoices ===
Status: 200
Total Records: 0
Sample: []


In [52]:
from pathlib import Path
import subprocess

project = Path("/content/procurement-management")

print("=== DATABASE FILES ===")

for db in project.rglob("*.sqlite"):
    print(db, "| Size:", db.stat().st_size)

print("\n=== CDS DATABASE CONFIGURATION ===")

result = subprocess.run(
    ["npx", "cds", "env", "requires.db"],
    cwd=project,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

=== DATABASE FILES ===
/content/procurement-management/db.sqlite | Size: 36864
/content/procurement-management/db-before-invoices.sqlite | Size: 36864
/content/procurement-management/db-backup.sqlite | Size: 36864

=== CDS DATABASE CONFIGURATION ===
{
  "impl": "@cap-js/sqlite",
  "credentials": {
    "url": ":memory:"
  },
  "pool": {
    "evictionRunIntervalMillis": 0,
    "min": 1,
    "max": 1
  },
  "data": [
    "db/data",
    "db/csv",
    "test/data"
  ],
  "kind": "sqlite"
}




In [53]:
import requests
from datetime import date

base = "http://127.0.0.1:4004/odata/v4/procurement"

r = requests.get(base + "/PurchaseOrders", timeout=10)
r.raise_for_status()

orders = r.json()["value"]

# Choose an existing purchase order
po = next(
    (
        o for o in orders
        if o["material"] == "Test Monitor"
        and int(o["quantity"]) >= 5
    ),
    None
)

if po is None:
    raise ValueError("No suitable Test Monitor purchase order found")

po_id = po["ID"]

print("Selected Purchase Order:", po_id)

receipt = requests.post(
    base + "/GoodsReceipts",
    json={
        "purchaseOrder_ID": po_id,
        "receivedQuantity": 5,
        "receiptDate": date.today().isoformat()
    },
    timeout=10
)

print("Goods Receipt Status:", receipt.status_code)
print(receipt.text[:1500])

Selected Purchase Order: 4219b0b7-a668-4a8f-8845-689fec32a83d
Goods Receipt Status: 201
{"@odata.context":"$metadata#GoodsReceipts/$entity","ID":"9a16cbee-5097-4e35-bd4b-b80469a3a65a","purchaseOrder_ID":"4219b0b7-a668-4a8f-8845-689fec32a83d","receivedQuantity":5,"receiptDate":"2026-11-21"}


In [54]:
for entity in ["GoodsReceipts", "Inventory"]:
    r = requests.get(base + "/" + entity, timeout=10)

    print("\n", entity)
    print("Status:", r.status_code)
    print(r.text[:1500])


 GoodsReceipts
Status: 200
{"@odata.context":"$metadata#GoodsReceipts","value":[{"ID":"9a16cbee-5097-4e35-bd4b-b80469a3a65a","purchaseOrder_ID":"4219b0b7-a668-4a8f-8845-689fec32a83d","receivedQuantity":5,"receiptDate":"2026-11-21"}]}

 Inventory
Status: 200
{"@odata.context":"$metadata#Inventory","value":[{"ID":"5b42cb16-8c43-4166-869c-d22a4e51a006","material":"Test Monitor","quantity":5,"lastUpdated":"2026-11-21"}]}


In [55]:
import requests
import uuid

base = "http://127.0.0.1:4004/odata/v4/procurement"

po_id = "4219b0b7-a668-4a8f-8845-689fec32a83d"

invoice_data = {
    "purchaseOrder_ID": po_id,
    "invoiceNumber": "INV-" + uuid.uuid4().hex[:8],
    "invoiceQuantity": 5,
    "invoiceAmount": 4000
}

response = requests.post(
    base + "/SupplierInvoices",
    json=invoice_data,
    timeout=10
)

print("Invoice Status:", response.status_code)
print("Invoice Details:", response.text[:1500])

Invoice Status: 201
Invoice Details: {"@odata.context":"$metadata#SupplierInvoices/$entity","ID":"08147693-cb69-4d7b-8e0e-e6cd82a8b424","purchaseOrder_ID":"4219b0b7-a668-4a8f-8845-689fec32a83d","invoiceNumber":"INV-fb2adba5","invoiceQuantity":5,"invoiceAmount":"4000.00","status":"Matched"}


In [56]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

response = requests.get(
    base + "/SupplierInvoices",
    timeout=10
)

print("Status:", response.status_code)

response.raise_for_status()
invoices = response.json()["value"]

print("Total Invoices:", len(invoices))

for invoice in invoices:
    print("Invoice Number:", invoice["invoiceNumber"])
    print("Quantity:", invoice["invoiceQuantity"])
    print("Amount:", invoice["invoiceAmount"])
    print("Status:", invoice["status"])

Status: 200
Total Invoices: 1
Invoice Number: INV-fb2adba5
Quantity: 5
Amount: 4000.00
Status: Matched


In [57]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

r = requests.get(base + "/SupplierInvoices", timeout=10)
r.raise_for_status()

for invoice in r.json()["value"]:
    print("Invoice:", invoice["invoiceNumber"])
    print("Amount:", invoice["invoiceAmount"])
    print("Status:", invoice["status"])

print("Total Invoices:", len(r.json()["value"]))

Invoice: INV-fb2adba5
Amount: 4000.00
Status: Matched
Total Invoices: 1


In [58]:
!pip -q install gradio pandas plotly requests

In [59]:
import requests
import pandas as pd
import plotly.express as px
import gradio as gr

BASE = "http://127.0.0.1:4004/odata/v4/procurement"

def fetch_data(entity):
    response = requests.get(f"{BASE}/{entity}", timeout=10)
    response.raise_for_status()
    return pd.DataFrame(response.json().get("value", []))

def dashboard():
    po = fetch_data("PurchaseOrders")
    gr_data = fetch_data("GoodsReceipts")
    inv = fetch_data("Inventory")
    invoices = fetch_data("SupplierInvoices")

    # KPIs
    total_orders = len(po)

    if not po.empty:
        po["quantity"] = pd.to_numeric(po["quantity"])
        po["unitPrice"] = pd.to_numeric(po["unitPrice"])
        po["orderValue"] = po["quantity"] * po["unitPrice"]
        total_value = po["orderValue"].sum()
        total_ordered = po["quantity"].sum()
    else:
        total_value = 0
        total_ordered = 0

    total_received = (
        pd.to_numeric(gr_data["receivedQuantity"]).sum()
        if not gr_data.empty else 0
    )

    matched_invoices = (
        len(invoices[invoices["status"] == "Matched"])
        if not invoices.empty else 0
    )

    fulfillment = (
        total_received / total_ordered * 100
        if total_ordered else 0
    )

    summary = f"""
    ## Procurement Performance

    | KPI | Value |
    |---|---:|
    | Purchase Orders | {total_orders} |
    | Total Purchase Value | {total_value:,.2f} SAR |
    | Units Ordered | {total_ordered} |
    | Units Received | {total_received} |
    | Fulfillment Rate | {fulfillment:.1f}% |
    | Matched Invoices | {matched_invoices} |
    """

    # Purchase order chart
    if not po.empty:
        order_chart = px.bar(
            po,
            x="material",
            y="orderValue",
            title="Purchase Order Value by Material",
            labels={
                "material": "Material",
                "orderValue": "Value (SAR)"
            }
        )
    else:
        order_chart = px.bar(title="No Purchase Orders")

    # Inventory chart
    if not inv.empty:
        inv["quantity"] = pd.to_numeric(inv["quantity"])
        inventory_chart = px.bar(
            inv,
            x="material",
            y="quantity",
            title="Current Inventory",
            labels={
                "material": "Material",
                "quantity": "Quantity"
            }
        )
    else:
        inventory_chart = px.bar(title="No Inventory Data")

    return (
        summary,
        order_chart,
        inventory_chart,
        po,
        gr_data,
        inv,
        invoices
    )

with gr.Blocks(title="SAP Procurement Dashboard") as demo:

    gr.Markdown(
        "# SAP Procurement & Inventory Dashboard"
    )

    gr.Markdown(
        "SAP CAP | OData Services | Procurement Analytics"
    )

    refresh = gr.Button("Refresh Dashboard")

    summary = gr.Markdown()

    with gr.Row():
        orders_chart = gr.Plot()
        inventory_chart = gr.Plot()

    with gr.Tab("Purchase Orders"):
        po_table = gr.Dataframe(interactive=False)

    with gr.Tab("Goods Receipts"):
        gr_table = gr.Dataframe(interactive=False)

    with gr.Tab("Inventory"):
        inv_table = gr.Dataframe(interactive=False)

    with gr.Tab("Supplier Invoices"):
        invoice_table = gr.Dataframe(interactive=False)

    outputs = [
        summary,
        orders_chart,
        inventory_chart,
        po_table,
        gr_table,
        inv_table,
        invoice_table
    ]

    refresh.click(
        fn=dashboard,
        outputs=outputs
    )

    demo.load(
        fn=dashboard,
        outputs=outputs
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://240b69929ccda65dc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [60]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

entities = [
    "PurchaseOrders",
    "GoodsReceipts",
    "Inventory",
    "SupplierInvoices"
]

before = {}

for entity in entities:
    r = requests.get(f"{base}/{entity}", timeout=10)
    r.raise_for_status()
    before[entity] = len(r.json()["value"])

print("Records before restart:", before)

Records before restart: {'PurchaseOrders': 2, 'GoodsReceipts': 1, 'Inventory': 1, 'SupplierInvoices': 1}


In [61]:
after = {}

for entity in entities:
    r = requests.get(f"{base}/{entity}", timeout=10)
    r.raise_for_status()
    after[entity] = len(r.json()["value"])

print("Before:", before)
print("After: ", after)

if before == after:
    print("PASS: Data persisted after server restart")
else:
    print("CHECK: Record counts changed after restart")

Before: {'PurchaseOrders': 2, 'GoodsReceipts': 1, 'Inventory': 1, 'SupplierInvoices': 1}
After:  {'PurchaseOrders': 2, 'GoodsReceipts': 1, 'Inventory': 1, 'SupplierInvoices': 1}
PASS: Data persisted after server restart


In [62]:
import requests
from datetime import date

base = "http://127.0.0.1:4004/odata/v4/procurement"

po_id = "4219b0b7-a668-4a8f-8845-689fec32a83d"

response = requests.post(
    base + "/GoodsReceipts",
    json={
        "purchaseOrder_ID": po_id,
        "receivedQuantity": 20,
        "receiptDate": date.today().isoformat()
    },
    timeout=10
)

print("Test Status:", response.status_code)
print("Response:", response.text[:1000])

if response.status_code == 400:
    print("PASS: Over-receipt rejected")
else:
    print("CHECK: Review goods receipt validation")

Test Status: 400
Response: {"error":{"message":"Goods receipt exceeds ordered quantity","code":"400","@Common.numericSeverity":4}}
PASS: Over-receipt rejected


In [63]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

# Get an existing matched invoice
r = requests.get(base + "/SupplierInvoices", timeout=10)
r.raise_for_status()

matched = [
    i for i in r.json()["value"]
    if i["status"] == "Matched"
]

if not matched:
    print("No matched invoice found")
else:
    existing = matched[0]

    duplicate = {
        "purchaseOrder_ID": existing["purchaseOrder_ID"],
        "invoiceNumber": existing["invoiceNumber"],
        "invoiceQuantity": existing["invoiceQuantity"],
        "invoiceAmount": existing["invoiceAmount"]
    }

    response = requests.post(
        base + "/SupplierInvoices",
        json=duplicate,
        timeout=10
    )

    print("Duplicate Invoice Status:", response.status_code)
    print("Response:", response.text[:1000])

Duplicate Invoice Status: 400
Response: {"error":{"message":"Invoice quantity exceeds uninvoiced received quantity","code":"400","@Common.numericSeverity":4}}


In [64]:
from pathlib import Path
import shutil

file = Path(
    "/content/procurement-management/srv/procurement-service.js"
)

code = file.read_text()

# Backup current implementation
shutil.copy2(
    file,
    file.with_name("procurement-service.before-duplicate.js.txt")
)

duplicate_check = """
    // Prevent duplicate supplier invoice numbers
    const duplicateInvoice = await tx.run(
      SELECT.one.from(SupplierInvoices)
        .where({ invoiceNumber: req.data.invoiceNumber })
    );

    if (duplicateInvoice) {
      return req.reject(
        409,
        'Duplicate invoice number'
      );
    }

"""

anchor = "    const order = await tx.run("

if "Duplicate invoice number" not in code:
    if anchor not in code:
        raise ValueError(
            "Could not find invoice validation insertion point"
        )

    # Insert specifically within the SupplierInvoices handler
    handler_start = code.find(
        "srv.before('CREATE', 'SupplierInvoices'"
    )

    if handler_start == -1:
        raise ValueError("SupplierInvoices handler not found")

    position = code.find(anchor, handler_start)

    if position == -1:
        raise ValueError("Invoice order lookup not found")

    code = (
        code[:position]
        + duplicate_check
        + code[position:]
    )

    file.write_text(code)

print("Duplicate invoice validation added successfully")

Duplicate invoice validation added successfully


In [65]:
import subprocess
import time

project = "/content/procurement-management"

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-1200:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: ':memory:' }
/> successfully deployed to in-memory database. 

[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 894 ms
(node:17689) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [66]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

r = requests.get(base + "/SupplierInvoices", timeout=10)
r.raise_for_status()

matched = [
    i for i in r.json()["value"]
    if i["status"] == "Matched"
]

if not matched:
    print("No matched invoice found")
else:
    existing = matched[0]

    response = requests.post(
        base + "/SupplierInvoices",
        json={
            "purchaseOrder_ID": existing["purchaseOrder_ID"],
            "invoiceNumber": existing["invoiceNumber"],
            "invoiceQuantity": existing["invoiceQuantity"],
            "invoiceAmount": existing["invoiceAmount"]
        },
        timeout=10
    )

    print("Duplicate Invoice Status:", response.status_code)
    print("Response:", response.text[:1000])

    if (
        response.status_code == 409
        and "Duplicate invoice number" in response.text
    ):
        print("PASS: Duplicate invoice rejected correctly")
    else:
        print("CHECK: Duplicate validation needs review")

No matched invoice found


In [67]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

r = requests.get(
    base + "/SupplierInvoices",
    timeout=10
)

print("Status:", r.status_code)
print("Invoices:", r.text[:2000])

Status: 200
Invoices: {"@odata.context":"$metadata#SupplierInvoices","value":[]}


In [68]:
import requests
import uuid

base = "http://127.0.0.1:4004/odata/v4/procurement"

def get_records(entity):
    r = requests.get(f"{base}/{entity}", timeout=10)
    r.raise_for_status()
    return r.json()["value"]

orders = get_records("PurchaseOrders")
receipts = get_records("GoodsReceipts")
invoices = get_records("SupplierInvoices")

selected = None

for po in orders:
    po_id = po["ID"]

    received = sum(
        int(g["receivedQuantity"])
        for g in receipts
        if g["purchaseOrder_ID"] == po_id
    )

    invoiced = sum(
        int(i["invoiceQuantity"])
        for i in invoices
        if i["purchaseOrder_ID"] == po_id
        and i["status"] == "Matched"
    )

    available = received - invoiced

    if available > 0:
        selected = (po, available)
        break

if selected is None:
    print("No uninvoiced received quantity available.")
    print("We need to inspect receipts before creating an invoice.")

else:
    po, available = selected

    invoice_data = {
        "purchaseOrder_ID": po["ID"],
        "invoiceNumber": "INV-" + uuid.uuid4().hex[:8],
        "invoiceQuantity": available,
        "invoiceAmount": round(
            available * float(po["unitPrice"]), 2
        )
    }

    # Create valid invoice
    first = requests.post(
        base + "/SupplierInvoices",
        json=invoice_data,
        timeout=10
    )

    print("First Invoice Status:", first.status_code)
    print("First Response:", first.text[:1000])

    # Test duplicate only if first invoice was saved
    if first.status_code == 201:

        duplicate = requests.post(
            base + "/SupplierInvoices",
            json=invoice_data,
            timeout=10
        )

        print("\nDuplicate Invoice Status:", duplicate.status_code)
        print("Duplicate Response:", duplicate.text[:1000])

        if (
            duplicate.status_code == 409
            and "Duplicate invoice number" in duplicate.text
        ):
            print("PASS: Duplicate invoice validation works")
        else:
            print("CHECK: Duplicate validation needs review")

No uninvoiced received quantity available.
We need to inspect receipts before creating an invoice.


In [69]:
import requests

base = "http://127.0.0.1:4004/odata/v4/procurement"

def get_data(entity):
    r = requests.get(f"{base}/{entity}", timeout=10)
    r.raise_for_status()
    return r.json()["value"]

orders = get_data("PurchaseOrders")
receipts = get_data("GoodsReceipts")
inventory = get_data("Inventory")
invoices = get_data("SupplierInvoices")

print("=== DATABASE SUMMARY ===")
print("Purchase Orders:", len(orders))
print("Goods Receipts:", len(receipts))
print("Inventory Records:", len(inventory))
print("Supplier Invoices:", len(invoices))

print("\n=== PURCHASE ORDER DETAILS ===")

for po in orders:
    po_id = po["ID"]

    received = sum(
        int(r["receivedQuantity"])
        for r in receipts
        if r["purchaseOrder_ID"] == po_id
    )

    invoiced = sum(
        int(i["invoiceQuantity"])
        for i in invoices
        if i["purchaseOrder_ID"] == po_id
        and i["status"] == "Matched"
    )

    print("\nMaterial:", po["material"])
    print("PO ID:", po_id)
    print("Ordered:", po["quantity"])
    print("Received:", received)
    print("Invoiced:", invoiced)
    print("Available to Invoice:", received - invoiced)

print("\n=== INVOICE DETAILS ===")

for invoice in invoices:
    print(
        invoice["invoiceNumber"],
        "| Status:", invoice["status"],
        "| Quantity:", invoice["invoiceQuantity"]
    )

=== DATABASE SUMMARY ===
Purchase Orders: 0
Goods Receipts: 0
Inventory Records: 0
Supplier Invoices: 0

=== PURCHASE ORDER DETAILS ===

=== INVOICE DETAILS ===


In [70]:
from pathlib import Path
import subprocess

project = Path("/content/procurement-management")

print("=== DATABASE FILES ===")

for file in project.rglob("*.sqlite"):
    print(
        "File:", file,
        "| Size:", file.stat().st_size,
        "bytes"
    )

print("\n=== DATABASE CONFIGURATION ===")

result = subprocess.run(
    ["npx", "cds", "env", "requires.db"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

print("\n=== PROJECT CONFIGURATION ===")

package = project / "package.json"

if package.exists():
    print(package.read_text()[:4000])

=== DATABASE FILES ===
File: /content/procurement-management/db.sqlite | Size: 36864 bytes
File: /content/procurement-management/db-before-invoices.sqlite | Size: 36864 bytes
File: /content/procurement-management/db-backup.sqlite | Size: 36864 bytes

=== DATABASE CONFIGURATION ===
{
  "impl": "@cap-js/sqlite",
  "credentials": {
    "url": ":memory:"
  },
  "pool": {
    "evictionRunIntervalMillis": 0,
    "min": 1,
    "max": 1
  },
  "data": [
    "db/data",
    "db/csv",
    "test/data"
  ],
  "kind": "sqlite"
}



=== PROJECT CONFIGURATION ===
{
  "devDependencies": {
    "@cap-js/sqlite": "^3.1.3"
  }
}



In [71]:
import sqlite3
from pathlib import Path

project = Path("/content/procurement-management")

for db_file in project.rglob("*.sqlite"):

    print("\nDATABASE:", db_file)

    try:
        # Read-only: don't create or modify database files
        connection = sqlite3.connect(
            f"file:{db_file}?mode=ro",
            uri=True
        )

        tables = connection.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """).fetchall()

        for (table,) in tables:
            if table.startswith("procurement_"):
                count = connection.execute(
                    f'SELECT COUNT(*) FROM "{table}"'
                ).fetchone()[0]

                print(table, ":", count, "records")

        connection.close()

    except Exception as e:
        print("Error:", e)


DATABASE: /content/procurement-management/db.sqlite
procurement_GoodsReceipts : 0 records
procurement_Inventory : 0 records
procurement_PurchaseOrders : 0 records
procurement_SupplierInvoices : 0 records
procurement_Suppliers : 0 records

DATABASE: /content/procurement-management/db-before-invoices.sqlite
procurement_GoodsReceipts : 0 records
procurement_PurchaseOrders : 0 records
procurement_Suppliers : 0 records

DATABASE: /content/procurement-management/db-backup.sqlite
procurement_GoodsReceipts : 0 records
procurement_PurchaseOrders : 0 records
procurement_Suppliers : 0 records


In [72]:
from pathlib import Path
import json
import shutil

project = Path("/content/procurement-management")
package_file = project / "package.json"

# Backup package configuration
shutil.copy2(
    package_file,
    project / "package-before-db-fix.json"
)

package = json.loads(package_file.read_text())

package.setdefault("cds", {})
package["cds"].setdefault("requires", {})

package["cds"]["requires"]["db"] = {
    "kind": "sqlite",
    "credentials": {
        "url": "db.sqlite"
    }
}

package_file.write_text(
    json.dumps(package, indent=2) + "\n"
)

print("Database configuration updated")
print(json.dumps(package["cds"]["requires"]["db"], indent=2))

Database configuration updated
{
  "kind": "sqlite",
  "credentials": {
    "url": "db.sqlite"
  }
}


In [73]:
import subprocess
import time

project = "/content/procurement-management"

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(6)

print("Server running:", server.poll() is None)
print(open("/content/cap-server.log").read()[-2000:])

Server running: True
[cds] - loaded model from 2 file(s):
 
  srv/procurement-service.cds
  db/schema.cds

[cds] - connect to db > sqlite { url: 'db.sqlite' }
[cds] - using auth strategy { kind: 'mocked' }
[cds] - serving ProcurementService {
  at: [ '/odata/v4/procurement' ],
  decl: 'srv/procurement-service.cds:3',
  impl: 'srv/procurement-service.js'
}
[cds] - server listening on { url: 'http://localhost:4004' }
[cds] - server v10.2.2 launched in 626 ms
(node:18342) ExperimentalWarning: SQLite is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)



In [74]:
import subprocess

result = subprocess.run(
    ["npx", "cds", "env", "requires.db"],
    cwd="/content/procurement-management",
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

{
  "impl": "@cap-js/sqlite",
  "credentials": {
    "url": "db.sqlite"
  },
  "pool": {
    "evictionRunIntervalMillis": 0,
    "min": 1,
    "max": 1
  },
  "data": [
    "db/data",
    "db/csv",
    "test/data"
  ],
  "kind": "sqlite"
}




In [75]:
import requests
import uuid

base = "http://127.0.0.1:4004/odata/v4/procurement"

test_name = "Persistence-Test-" + uuid.uuid4().hex[:8]

response = requests.post(
    base + "/Suppliers",
    json={
        "name": test_name,
        "email": "persistence@example.com"
    },
    timeout=10
)

print("Create Status:", response.status_code)
print("Response:", response.text[:1000])

response.raise_for_status()

supplier_id = response.json()["ID"]

print("Test Supplier ID:", supplier_id)

Create Status: 201
Response: {"@odata.context":"$metadata#Suppliers/$entity","ID":"1df8a82c-f32d-43ca-96a1-d75e43bd0ee3","name":"Persistence-Test-0e0dcf74","email":"persistence@example.com"}
Test Supplier ID: 1df8a82c-f32d-43ca-96a1-d75e43bd0ee3


In [76]:
import sqlite3

db_path = "/content/procurement-management/db.sqlite"

with sqlite3.connect(db_path) as conn:
    result = conn.execute(
        'SELECT ID, name FROM "procurement_Suppliers" WHERE ID = ?',
        (supplier_id,)
    ).fetchone()

print("Supplier in SQLite:", result)

if result:
    print("PASS: API data saved to db.sqlite")
else:
    print("CHECK: API is using a different database")

Supplier in SQLite: ('1df8a82c-f32d-43ca-96a1-d75e43bd0ee3', 'Persistence-Test-0e0dcf74')
PASS: API data saved to db.sqlite


In [77]:
import subprocess
import time
import requests

project = "/content/procurement-management"

# Stop the existing server
subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

time.sleep(2)

# Restart without redeploying the database
log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(6)

response = requests.get(
    "http://127.0.0.1:4004/odata/v4/procurement/Suppliers",
    timeout=10
)

response.raise_for_status()

suppliers = response.json()["value"]

found = any(
    s["ID"] == supplier_id
    for s in suppliers
)

print("Server Running:", server.poll() is None)
print("Supplier Found After Restart:", found)

if found:
    print("PASS: Database persistence verified")
else:
    print("FAIL: Supplier disappeared after restart")

Server Running: True
Supplier Found After Restart: True
PASS: Database persistence verified


In [78]:
import requests
import uuid
from datetime import date

BASE = "http://127.0.0.1:4004/odata/v4/procurement"

def create(entity, data):
    r = requests.post(
        f"{BASE}/{entity}",
        json=data,
        timeout=10
    )
    print(f"{entity}: {r.status_code}")
    print(r.text[:500])
    r.raise_for_status()
    return r.json()

# 1. Create supplier
supplier = create("Suppliers", {
    "name": "DXC Demo Supplier",
    "email": "demo@example.com"
})

# 2. Create purchase order
po = create("PurchaseOrders", {
    "supplier_ID": supplier["ID"],
    "material": "Dell Laptop",
    "quantity": 10,
    "unitPrice": 3500,
    "status": "Ordered"
})

# 3. Receive six laptops
receipt = create("GoodsReceipts", {
    "purchaseOrder_ID": po["ID"],
    "receivedQuantity": 6,
    "receiptDate": date.today().isoformat()
})

# 4. Create matching invoice
invoice_number = "INV-" + uuid.uuid4().hex[:8]

invoice_data = {
    "purchaseOrder_ID": po["ID"],
    "invoiceNumber": invoice_number,
    "invoiceQuantity": 6,
    "invoiceAmount": 21000
}

invoice = create("SupplierInvoices", invoice_data)

print("\nInvoice Status:", invoice.get("status"))

# 5. Test duplicate invoice
duplicate = requests.post(
    BASE + "/SupplierInvoices",
    json=invoice_data,
    timeout=10
)

print("\nDuplicate Invoice Status:", duplicate.status_code)
print("Duplicate Response:", duplicate.text[:1000])

if (
    duplicate.status_code == 409
    and "Duplicate invoice number" in duplicate.text
):
    print("PASS: Duplicate invoice rejected")
else:
    print("CHECK: Duplicate invoice validation")

Suppliers: 201
{"@odata.context":"$metadata#Suppliers/$entity","ID":"e1dffd58-00d9-4b62-a446-84eb74b9c7f4","name":"DXC Demo Supplier","email":"demo@example.com"}
PurchaseOrders: 201
{"@odata.context":"$metadata#PurchaseOrders/$entity","ID":"99b6c432-10f8-400e-b418-e50660526100","supplier_ID":"e1dffd58-00d9-4b62-a446-84eb74b9c7f4","material":"Dell Laptop","quantity":10,"unitPrice":"3500.00","status":"Ordered"}
GoodsReceipts: 201
{"@odata.context":"$metadata#GoodsReceipts/$entity","ID":"e3863993-4ca9-4d98-a987-d877836881b1","purchaseOrder_ID":"99b6c432-10f8-400e-b418-e50660526100","receivedQuantity":6,"receiptDate":"2026-11-21"}
SupplierInvoices: 201
{"@odata.context":"$metadata#SupplierInvoices/$entity","ID":"d61eccd2-c4ff-44bc-861d-9e3fcb869f41","purchaseOrder_ID":"99b6c432-10f8-400e-b418-e50660526100","invoiceNumber":"INV-81d16d9d","invoiceQuantity":6,"invoiceAmount":"21000.00","status":"Matched"}

Invoice Status: Matched

Duplicate Invoice Status: 409
Duplicate Response: {"error":{"m

In [80]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [81]:
import shutil
from pathlib import Path
from datetime import datetime

project = Path("/content/procurement-management")
backup_dir = Path("/content/drive/MyDrive/SAP_CAP_Project")

backup_dir.mkdir(parents=True, exist_ok=True)

backup_name = (
    "SAP_Procurement_Backup_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

backup_path = backup_dir / backup_name

shutil.copytree(
    project,
    backup_path,
    ignore=shutil.ignore_patterns(
        "node_modules",
        ".git",
        ".cache"
    )
)

print("Backup completed!")
print("Location:", backup_path)

Backup completed!
Location: /content/drive/MyDrive/SAP_CAP_Project/SAP_Procurement_Backup_20261121_162437


In [82]:
from pathlib import Path

project = Path("/content/procurement-management")

readme = """# SAP Procurement & Inventory Management

## Project Overview
A procurement and inventory management prototype
developed using SAP Cloud Application Programming Model (CAP).

The project demonstrates procurement workflows,
business rule validation, and OData-based integration.

## Technology Stack
- SAP CAP (Node.js)
- SAP CDS
- OData V4
- SQLite
- JavaScript
- Python
- Gradio
- Plotly

## Key Features
1. Supplier Management
2. Purchase Order Management
3. Goods Receipt Validation
4. Automated Inventory Updates
5. Three-Way Invoice Matching
6. Duplicate Invoice Validation
7. Procurement Analytics Dashboard

## Business Process
Supplier → Purchase Order → Goods Receipt
→ Inventory Update → Invoice Matching

## API Services
- /odata/v4/procurement/Suppliers
- /odata/v4/procurement/PurchaseOrders
- /odata/v4/procurement/GoodsReceipts
- /odata/v4/procurement/Inventory
- /odata/v4/procurement/SupplierInvoices

## Business Value
- Reduce manual procurement errors
- Improve inventory visibility
- Validate supplier invoices
- Support procurement reporting
- Demonstrate SAP CAP extensibility

## Technical Scope
This is a standalone SAP CAP prototype.
It is not an SAP S/4HANA deployment or integration.

## Author
Eman Mubarak
Computer Engineering
"""

(project / "README.md").write_text(readme, encoding="utf-8")

print("README.md created successfully!")

README.md created successfully!


In [83]:
required_files = [
    "package.json",
    "db/schema.cds",
    "srv/procurement-service.cds",
    "srv/procurement-service.js",
    "README.md"
]

for filename in required_files:
    path = project / filename
    print(
        "PASS" if path.exists() else "MISSING",
        filename
    )

PASS package.json
PASS db/schema.cds
PASS srv/procurement-service.cds
PASS srv/procurement-service.js
PASS README.md


In [84]:
from pathlib import Path

project = Path("/content/procurement-management")
dashboard_dir = project / "dashboard"
dashboard_dir.mkdir(exist_ok=True)

print("Dashboard folder created")

Dashboard folder created


In [85]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://240b69929ccda65dc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [86]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://240b69929ccda65dc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [87]:
%%writefile /content/procurement-management/dashboard/app.py
(project / "dashboard/requirements.txt").write_text(
    "gradio\npandas\nplotly\nrequests\n",
    encoding="utf-8"
)

print("Dashboard requirements created")

Writing /content/procurement-management/dashboard/app.py


In [88]:
files = [
    "package.json",
    "db/schema.cds",
    "srv/procurement-service.cds",
    "srv/procurement-service.js",
    "README.md",
    "dashboard/app.py",
    "dashboard/requirements.txt"
]

print("=== PROJECT READINESS ===")

for filename in files:
    path = project / filename
    print(
        "PASS" if path.exists() else "MISSING",
        filename
    )

=== PROJECT READINESS ===
PASS package.json
PASS db/schema.cds
PASS srv/procurement-service.cds
PASS srv/procurement-service.js
PASS README.md
PASS dashboard/app.py
MISSING dashboard/requirements.txt


In [89]:
from pathlib import Path

project = Path("/content/procurement-management")

requirements = project / "dashboard" / "requirements.txt"

requirements.parent.mkdir(parents=True, exist_ok=True)

requirements.write_text(
    "gradio\n"
    "pandas\n"
    "plotly\n"
    "requests\n",
    encoding="utf-8"
)

print("PASS:", requirements.exists())
print("\nInstalled requirements:")
print(requirements.read_text())

PASS: True

Installed requirements:
gradio
pandas
plotly
requests



In [90]:
from pathlib import Path

project = Path("/content/procurement-management")

requirements = project / "dashboard" / "requirements.txt"

requirements.parent.mkdir(parents=True, exist_ok=True)

requirements.write_text(
    "gradio\n"
    "pandas\n"
    "plotly\n"
    "requests\n",
    encoding="utf-8"
)

print("PASS:", requirements.exists())
print("\nInstalled requirements:")
print(requirements.read_text())

PASS: True

Installed requirements:
gradio
pandas
plotly
requests



In [91]:
from pathlib import Path
from datetime import datetime
import requests

project = Path("/content/procurement-management")
base = "http://127.0.0.1:4004/odata/v4/procurement"

services = [
    "Suppliers",
    "PurchaseOrders",
    "GoodsReceipts",
    "Inventory",
    "SupplierInvoices"
]

results = []

for service in services:
    try:
        r = requests.get(f"{base}/{service}", timeout=10)
        passed = r.status_code == 200

        results.append({
            "test": f"{service} API",
            "status": "PASS" if passed else "FAIL",
            "details": f"HTTP {r.status_code}"
        })
    except requests.RequestException as e:
        results.append({
            "test": f"{service} API",
            "status": "FAIL",
            "details": str(e)
        })

files = [
    "package.json",
    "db/schema.cds",
    "srv/procurement-service.cds",
    "srv/procurement-service.js",
    "dashboard/app.py",
    "dashboard/requirements.txt",
    "README.md"
]

for name in files:
    exists = (project / name).is_file()
    results.append({
        "test": name,
        "status": "PASS" if exists else "FAIL",
        "details": "File exists" if exists else "Missing"
    })

report = [
    "# SAP CAP Project Test Report",
    "",
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    "",
    "| Test | Result | Details |",
    "|---|---|---|"
]

for result in results:
    report.append(
        f"| {result['test']} | {result['status']} | {result['details']} |"
    )

report_path = project / "TEST_REPORT.md"
report_path.write_text("\n".join(report) + "\n", encoding="utf-8")

for result in results:
    print(result["status"], result["test"])

print("\nReport saved:", report_path)
print("Passed:", sum(r["status"] == "PASS" for r in results))
print("Total:", len(results))

PASS Suppliers API
PASS PurchaseOrders API
PASS GoodsReceipts API
PASS Inventory API
PASS SupplierInvoices API
PASS package.json
PASS db/schema.cds
PASS srv/procurement-service.cds
PASS srv/procurement-service.js
PASS dashboard/app.py
PASS dashboard/requirements.txt
PASS README.md

Report saved: /content/procurement-management/TEST_REPORT.md
Passed: 12
Total: 12


In [92]:
from pathlib import Path
import zipfile

project = Path("/content/procurement-management")
zip_path = Path("/content/SAP_Procurement_CAP_Project.zip")

excluded_dirs = {
    "node_modules", ".git", "__pycache__", ".cache"
}

excluded_suffixes = {
    ".sqlite", ".sqlite-shm", ".sqlite-wal",
    ".log", ".pyc"
}

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zipf:

    for file in project.rglob("*"):
        if not file.is_file():
            continue

        relative = file.relative_to(project)

        if any(p in excluded_dirs for p in relative.parts):
            continue

        if any(file.name.endswith(s) for s in excluded_suffixes):
            continue

        if "backup" in file.name.lower():
            continue

        zipf.write(
            file,
            arcname=str(Path("SAP_Procurement_CAP_Project") / relative)
        )

print("ZIP Created:", zip_path)
print("ZIP Size:", round(zip_path.stat().st_size / 1024, 2), "KB")

ZIP Created: /content/SAP_Procurement_CAP_Project.zip
ZIP Size: 16.18 KB


In [93]:
with zipfile.ZipFile(zip_path) as z:
    files = z.namelist()

    print("=== ZIP CONTENTS ===")

    for file in files:
        print(file)

    print("\nTotal Files:", len(files))

    bad = z.testzip()
    print("ZIP Integrity:", "PASS" if bad is None else "FAIL")

=== ZIP CONTENTS ===
SAP_Procurement_CAP_Project/.gitignore
SAP_Procurement_CAP_Project/package-before-db-fix.json
SAP_Procurement_CAP_Project/readme.md
SAP_Procurement_CAP_Project/README.md
SAP_Procurement_CAP_Project/package-lock.json
SAP_Procurement_CAP_Project/TEST_REPORT.md
SAP_Procurement_CAP_Project/package.json
SAP_Procurement_CAP_Project/.vscode/tasks.json
SAP_Procurement_CAP_Project/.gradio/certificate.pem
SAP_Procurement_CAP_Project/srv/procurement-service.js
SAP_Procurement_CAP_Project/srv/procurement-service.cds
SAP_Procurement_CAP_Project/srv/procurement-service.before-duplicate.js.txt
SAP_Procurement_CAP_Project/dashboard/requirements.txt
SAP_Procurement_CAP_Project/dashboard/app.py
SAP_Procurement_CAP_Project/db/schema.cds

Total Files: 15
ZIP Integrity: PASS


In [94]:
import shutil
from pathlib import Path

destination = Path(
    "/content/drive/MyDrive/SAP_CAP_Project/"
    "SAP_Procurement_CAP_Project.zip"
)

destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(zip_path, destination)

print("Saved to Google Drive:", destination)

Saved to Google Drive: /content/drive/MyDrive/SAP_CAP_Project/SAP_Procurement_CAP_Project.zip


In [95]:
from pathlib import Path

project = Path("/content/procurement-management")

folders = [
    "app",
    "app/procurement",
    "app/procurement/webapp"
]

for folder in folders:
    (project / folder).mkdir(parents=True, exist_ok=True)

print("SAP Fiori application structure created!")

for folder in folders:
    print("PASS:", folder)

SAP Fiori application structure created!
PASS: app
PASS: app/procurement
PASS: app/procurement/webapp


In [96]:
from pathlib import Path

project = Path("/content/procurement-management")

annotations = """
using ProcurementService as service
    from '../../srv/procurement-service';

annotate service.PurchaseOrders with @(
    UI.HeaderInfo: {
        TypeName: 'Purchase Order',
        TypeNamePlural: 'Purchase Orders',
        Title: { Value: material }
    },

    UI.SelectionFields: [
        material,
        status
    ],

    UI.LineItem: [
        { Value: material, Label: 'Material' },
        { Value: quantity, Label: 'Ordered Quantity' },
        { Value: unitPrice, Label: 'Unit Price' },
        { Value: status, Label: 'Status' }
    ],

    UI.Facets: [
        {
            $Type: 'UI.ReferenceFacet',
            Label: 'Purchase Order Details',
            Target: '@UI.FieldGroup#Details'
        }
    ],

    UI.FieldGroup #Details: {
        Data: [
            { Value: material },
            { Value: quantity },
            { Value: unitPrice },
            { Value: status }
        ]
    }
);

annotate service.Suppliers with @(
    UI.HeaderInfo: {
        TypeName: 'Supplier',
        TypeNamePlural: 'Suppliers',
        Title: { Value: name }
    },

    UI.LineItem: [
        { Value: name, Label: 'Supplier Name' },
        { Value: email, Label: 'Email Address' }
    ]
);

annotate service.Inventory with @(
    UI.HeaderInfo: {
        TypeName: 'Inventory Item',
        TypeNamePlural: 'Inventory'
    },

    UI.LineItem: [
        { Value: material, Label: 'Material' },
        { Value: quantity, Label: 'Stock Quantity' },
        { Value: lastUpdated, Label: 'Last Updated' }
    ]
);

annotate service.SupplierInvoices with @(
    UI.HeaderInfo: {
        TypeName: 'Supplier Invoice',
        TypeNamePlural: 'Supplier Invoices',
        Title: { Value: invoiceNumber }
    },

    UI.LineItem: [
        { Value: invoiceNumber },
        { Value: invoiceQuantity },
        { Value: invoiceAmount },
        { Value: status }
    ]
);
"""

file = project / "app/procurement/annotations.cds"
file.write_text(annotations, encoding="utf-8")

print("SAP Fiori Annotations created!")
print("File:", file)

SAP Fiori Annotations created!
File: /content/procurement-management/app/procurement/annotations.cds


In [97]:
import json

manifest = {
    "_version": "1.60.0",

    "sap.app": {
        "id": "procurement.management",
        "type": "application",
        "title": "Procurement Management",
        "description": "SAP CAP Procurement Management",
        "applicationVersion": {
            "version": "1.0.0"
        },
        "dataSources": {
            "mainService": {
                "uri": "/odata/v4/procurement/",
                "type": "OData",
                "settings": {
                    "odataVersion": "4.0"
                }
            }
        }
    },

    "sap.ui": {
        "technology": "UI5",
        "deviceTypes": {
            "desktop": True,
            "tablet": True,
            "phone": True
        }
    },

    "sap.ui5": {
        "dependencies": {
            "minUI5Version": "1.120.0",
            "libs": {
                "sap.m": {},
                "sap.fe.templates": {}
            }
        },

        "models": {
            "": {
                "dataSource": "mainService",
                "settings": {
                    "synchronizationMode": "None",
                    "operationMode": "Server",
                    "autoExpandSelect": True
                }
            }
        },

        "routing": {
            "routes": [
                {
                    "pattern": ":?query:",
                    "name": "PurchaseOrdersList",
                    "target": "PurchaseOrdersList"
                },
                {
                    "pattern": "PurchaseOrders({key}):?query:",
                    "name": "PurchaseOrdersObjectPage",
                    "target": "PurchaseOrdersObjectPage"
                }
            ],
            "targets": {
                "PurchaseOrdersList": {
                    "type": "Component",
                    "id": "PurchaseOrdersList",
                    "name": "sap.fe.templates.ListReport",
                    "options": {
                        "settings": {
                            "contextPath": "/PurchaseOrders"
                        }
                    }
                },
                "PurchaseOrdersObjectPage": {
                    "type": "Component",
                    "id": "PurchaseOrdersObjectPage",
                    "name": "sap.fe.templates.ObjectPage",
                    "options": {
                        "settings": {
                            "contextPath": "/PurchaseOrders"
                        }
                    }
                }
            }
        }
    }
}

manifest_file = (
    project / "app/procurement/webapp/manifest.json"
)

manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("SAP Fiori manifest created!")

SAP Fiori manifest created!


In [98]:
import subprocess

result = subprocess.run(
    ["npx", "cds", "compile", "srv", "app", "--to", "json"],
    cwd="/content/procurement-management",
    capture_output=True,
    text=True
)

print("Compilation Status:", result.returncode)

if result.returncode == 0:
    print("PASS: SAP CDS compilation successful")
else:
    print("Compilation Errors:")
    print(result.stderr[-3000:])

Compilation Status: 0
PASS: SAP CDS compilation successful


In [99]:
from pathlib import Path

project = Path("/content/procurement-management")
webapp = project / "app/procurement/webapp"

webapp.mkdir(parents=True, exist_ok=True)

component_js = """
sap.ui.define([
    "sap/fe/core/AppComponent"
], function (AppComponent) {
    "use strict";

    return AppComponent.extend(
        "procurement.management.Component",
        {
            metadata: {
                manifest: "json"
            }
        }
    );
});
"""

(webapp / "Component.js").write_text(
    component_js,
    encoding="utf-8"
)

print("PASS: Component.js created")

PASS: Component.js created


In [100]:
index_html = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>SAP Procurement Management</title>

    <script
        id="sap-ui-bootstrap"
        src="https://ui5.sap.com/resources/sap-ui-core.js"
        data-sap-ui-theme="sap_horizon"
        data-sap-ui-libs="sap.m,sap.fe.templates"
        data-sap-ui-resourceroots='{"procurement.management":"./"}'
        data-sap-ui-compatVersion="edge"
        data-sap-ui-async="true">
    </script>

    <script>
        sap.ui.getCore().attachInit(function () {
            sap.ui.require([
                "sap/ui/core/ComponentContainer"
            ], function (ComponentContainer) {

                new ComponentContainer({
                    name: "procurement.management",
                    async: true,
                    height: "100%"
                }).placeAt("content");

            });
        });
    </script>
</head>

<body class="sapUiBody" id="content">
</body>
</html>
"""

(webapp / "index.html").write_text(
    index_html,
    encoding="utf-8"
)

print("PASS: index.html created")

PASS: index.html created


In [101]:
import json

manifest_file = webapp / "manifest.json"

manifest = json.loads(
    manifest_file.read_text(encoding="utf-8")
)

manifest["sap.ui5"]["rootView"] = {
    "viewName": "sap.fe.core.rootView.NavContainer",
    "type": "XML",
    "async": True,
    "id": "appRootView"
}

manifest["sap.ui5"]["routing"]["config"] = {
    "routerClass": "sap.f.routing.Router"
}

manifest["sap.ui5"]["routing"]["routes"][0]["target"] = [
    "PurchaseOrdersList"
]

manifest["sap.ui5"]["routing"]["routes"][1]["target"] = [
    "PurchaseOrdersObjectPage"
]

manifest["sap.ui5"]["routing"]["targets"][
    "PurchaseOrdersList"
]["options"]["settings"]["navigation"] = {
    "PurchaseOrders": {
        "detail": {
            "route": "PurchaseOrdersObjectPage"
        }
    }
}

manifest_file.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("PASS: Fiori manifest updated")

PASS: Fiori manifest updated


In [102]:
import subprocess
import time
import requests

project = "/content/procurement-management"

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=project,
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

try:
    response = requests.get(
        "http://127.0.0.1:4004/odata/v4/procurement/$metadata",
        timeout=10
    )
    print("SAP CAP Status:", response.status_code)
except Exception as e:
    print("Server error:", e)

print("\nServer log:")
print(open("/content/cap-server.log").read()[-1500:])

SAP CAP Status: 200

Server log:
is an experimental feature and might change at any time
(Use `node --trace-warnings ...` to show where the warning was created)
[odata] - GET /odata/v4/procurement/$metadata 
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

In [108]:
from google.colab import output

output.serve_kernel_port_as_window(4004)



Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [109]:
from google.colab import output

output.serve_kernel_port_as_iframe(4004)

<IPython.core.display.Javascript object>

In [110]:
import requests

base = "http://127.0.0.1:4004"

paths = {
    "SAP CAP": "/odata/v4/procurement/$metadata",
    "SAP Fiori": "/procurement/webapp/index.html"
}

for name, path in paths.items():
    try:
        response = requests.get(base + path, timeout=10)
        print(name, ":", response.status_code)
        print("Content-Type:", response.headers.get("Content-Type"))
    except Exception as e:
        print(name, "ERROR:", e)

SAP CAP : 200
Content-Type: application/xml; charset=utf-8
SAP Fiori : 200
Content-Type: text/html; charset=utf-8


In [111]:
from pathlib import Path
import requests

project = Path("/content/procurement-management")
base = "http://127.0.0.1:4004"

print("=== SAP FIORI PROJECT CHECK ===")

files = [
    "app/procurement/webapp/index.html",
    "app/procurement/webapp/Component.js",
    "app/procurement/webapp/manifest.json",
    "app/procurement/annotations.cds"
]

for name in files:
    print(
        "PASS" if (project / name).exists() else "MISSING",
        name
    )

print("\n=== SERVICE CHECK ===")

for path in [
    "/odata/v4/procurement/$metadata",
    "/procurement/webapp/index.html"
]:
    try:
        response = requests.get(base + path, timeout=10)
        print(response.status_code, path)
    except Exception as e:
        print("ERROR", path, str(e))

=== SAP FIORI PROJECT CHECK ===
PASS app/procurement/webapp/index.html
PASS app/procurement/webapp/Component.js
PASS app/procurement/webapp/manifest.json
PASS app/procurement/annotations.cds

=== SERVICE CHECK ===
200 /odata/v4/procurement/$metadata
200 /procurement/webapp/index.html


In [112]:
from pathlib import Path
import json

project = Path("/content/procurement-management")
webapp = project / "app/procurement/webapp"
manifest_path = webapp / "manifest.json"

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

pages = {
    "Suppliers": "Manage Suppliers",
    "PurchaseOrders": "Manage Purchase Orders",
    "GoodsReceipts": "Post Goods Receipts",
    "Inventory": "Inventory Overview",
    "SupplierInvoices": "Manage Supplier Invoices"
}

routing = manifest["sap.ui5"]["routing"]
routes = routing["routes"]
targets = routing["targets"]

# الاحتفاظ بصفحة Purchase Orders الرئيسية
for entity, title in pages.items():

    target_name = entity + "List"

    targets[target_name] = {
        "type": "Component",
        "id": target_name,
        "name": "sap.fe.templates.ListReport",
        "options": {
            "settings": {
                "contextPath": "/" + entity,
                "variantManagement": "Page"
            }
        }
    }

    if entity != "PurchaseOrders":
        routes.append({
            "pattern": entity + ":?query:",
            "name": target_name,
            "target": target_name
        })

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("PASS: Fiori pages configured")

for entity, title in pages.items():
    print(title, "→", entity)

PASS: Fiori pages configured
Manage Suppliers → Suppliers
Manage Purchase Orders → PurchaseOrders
Post Goods Receipts → GoodsReceipts
Inventory Overview → Inventory
Manage Supplier Invoices → SupplierInvoices


In [113]:
annotations_path = project / "app/procurement/annotations.cds"

extra_annotations = """

annotate service.GoodsReceipts with @(
    UI.HeaderInfo: {
        TypeName: 'Goods Receipt',
        TypeNamePlural: 'Goods Receipts'
    },

    UI.LineItem: [
        { Value: receivedQuantity, Label: 'Received Quantity' },
        { Value: receiptDate, Label: 'Receipt Date' }
    ],

    UI.SelectionFields: [
        receiptDate
    ]
);
"""

current = annotations_path.read_text(encoding="utf-8")

if "annotate service.GoodsReceipts with" not in current:
    annotations_path.write_text(
        current + extra_annotations,
        encoding="utf-8"
    )

print("PASS: Goods Receipts annotations added")

PASS: Goods Receipts annotations added


In [114]:
import subprocess
import time
import requests

subprocess.run(
    ["fuser", "-k", "4004/tcp"],
    capture_output=True
)

log = open("/content/cap-server.log", "w")

server = subprocess.Popen(
    ["npx", "cds", "serve", "--port", "4004"],
    cwd=str(project),
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(7)

try:
    r = requests.get(
        "http://127.0.0.1:4004/odata/v4/procurement/$metadata",
        timeout=10
    )
    print("SAP CAP:", r.status_code)
except Exception as e:
    print("ERROR:", e)

SAP CAP: 200


In [117]:
from google.colab import output

output.serve_kernel_port_as_iframe(4004)

<IPython.core.display.Javascript object>